In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import re
from pathlib import Path
from config import engine

def extract_uid(filename):
    """Extract uid from filenames like 'u01.txt' -> 'u01'"""
    match = re.search(r'u\d+', filename)
    return match.group() if match else None

print("✓ Setup complete")

✓ Setup complete


In [2]:
project_root = Path.cwd().parent

# relative path inside the project
folder = project_root / "dataset" / "archive (1)" / "dataset" / "dinning"

print("Project root:", project_root)
print("Dataset folder:", folder)
print("Folder exists:", folder.exists())

list(folder.glob("*.txt"))[:5]

Project root: d:\uni\databases\project
Dataset folder: d:\uni\databases\project\dataset\archive (1)\dataset\dinning
Folder exists: True


[WindowsPath('d:/uni/databases/project/dataset/archive (1)/dataset/dinning/u01.txt'),
 WindowsPath('d:/uni/databases/project/dataset/archive (1)/dataset/dinning/u02.txt'),
 WindowsPath('d:/uni/databases/project/dataset/archive (1)/dataset/dinning/u04.txt'),
 WindowsPath('d:/uni/databases/project/dataset/archive (1)/dataset/dinning/u05.txt'),
 WindowsPath('d:/uni/databases/project/dataset/archive (1)/dataset/dinning/u07.txt')]

In [3]:
data = []

for file in folder.glob("*.txt"):
    uid = extract_uid(file.name)

    with open(file, "r", encoding="utf-8") as f:
        lines = f.readlines()

    for line in lines:
        line = line.strip()
        if line:
            data.append({
                "uid": uid,
                "dinning_event": line
            })

df = pd.DataFrame(data)

df.head()

,uid,dinning_event
0,u01,"2013-01-06 17:42:49,53 Commons,Supper"
1,u01,"2013-01-07 09:32:57,Novack Cafe,Breakfast"
2,u01,"2013-01-07 14:16:07,Courtyard Cafe,Lunch"
3,u01,"2013-01-08 12:51:22,Courtyard Cafe,Lunch"
4,u01,"2013-01-09 13:46:44,King Arthur Flour Coffee B..."


In [4]:
# split dinning_event into datetime, location, meal_type
df[['datetime', 'location', 'meal_type']] = df['dinning_event'].str.split(',', expand=True)

# drop old combined column
df = df.drop(columns=['dinning_event'])

# optional cleanup: strip extra spaces
df['datetime'] = df['datetime'].str.strip()
df['location'] = df['location'].str.strip()
df['meal_type'] = df['meal_type'].str.strip()

# create meal_id
df.insert(0, 'meal_id', range(1, len(df) + 1))

print("✓ Parsed dataframe ready for SQL")
print("Shape after parsing:", df.shape)
print(df.head())


✓ Parsed dataframe ready for SQL
Shape after parsing: (7482, 5)
   meal_id  uid             datetime                      location  meal_type
0        1  u01  2013-01-06 17:42:49                    53 Commons     Supper
1        2  u01  2013-01-07 09:32:57                   Novack Cafe  Breakfast
2        3  u01  2013-01-07 14:16:07                Courtyard Cafe      Lunch
3        4  u01  2013-01-08 12:51:22                Courtyard Cafe      Lunch
4        5  u01  2013-01-09 13:46:44  King Arthur Flour Coffee Bar      Lunch


In [5]:
# insert into SQL table
df.to_sql(
    "dinning",
    engine,
    schema="public",
    if_exists="append",
    index=False
)

print("✓ Data inserted into table 'dinning'")


# verification
check_df = pd.read_sql("SELECT * FROM public.dinning LIMIT 10;", engine)
check_df

✓ Data inserted into table 'dinning'


,meal_id,uid,datetime,location,meal_type
0,1,u01,2013-01-06 17:42:49,53 Commons,Supper
1,2,u01,2013-01-07 09:32:57,Novack Cafe,Breakfast
2,3,u01,2013-01-07 14:16:07,Courtyard Cafe,Lunch
3,4,u01,2013-01-08 12:51:22,Courtyard Cafe,Lunch
4,5,u01,2013-01-09 13:46:44,King Arthur Flour Coffee Bar,Lunch
5,6,u01,2013-01-09 18:33:18,Collis Cafe,Supper
6,7,u01,2013-01-10 17:57:07,53 Commons,Supper
7,8,u01,2013-01-11 10:39:39,King Arthur Flour Coffee Bar,Snack
8,9,u01,2013-01-11 14:01:56,Courtyard Cafe,Lunch
9,10,u01,2013-01-15 11:58:29,Courtyard Cafe,Snack


In [3]:
# ===========================
# LOAD MISSING DINNING DATA
# ===========================

import os
import json

print("="*60)
print("LOADING MISSING DINNING DATA")
print("="*60)

dining_halls_dir = '/Users/test/Downloads/Data Management & Databases/studentlife_project/data/EMA/response/Dining Halls'

# Get already loaded students
loaded_dinning = pd.read_sql("SELECT DISTINCT uid FROM dinning", engine)
loaded_uids = set(loaded_dinning['uid'].tolist())

missing_students = ['u00', 'u03', 'u13', 'u17', 'u23', 'u31', 'u34', 'u35', 'u39', 'u41', 'u44', 'u45', 'u50', 'u51', 'u52', 'u53', 'u56', 'u58']

all_responses = []

for uid in missing_students:
    filename = f"Dining Halls_{uid}.json"
    filepath = os.path.join(dining_halls_dir, filename)
    
    if not os.path.exists(filepath):
        print(f"⚠️  File not found: {filename}")
        continue
    
    try:
        import json
        with open(filepath, 'r') as f:
            data = json.load(f)
        
        # Process each entry
        for entry in data:
            datetime_str = entry.get('datetime')
            location = entry.get('location')
            
            all_responses.append({
                'uid': uid,
                'datetime': datetime_str,
                'location': location
            })
        
        print(f"✓ Loaded {uid}: {len(data)} entries")
        
    except Exception as e:
        print(f"❌ Error loading {uid}: {e}")

# Insert into database
if all_responses:
    df = pd.DataFrame(all_responses)
    df.to_sql('dinning', engine, if_exists='append', index=False)
    print(f"\n✅ Loaded {len(df):,} dinning records from {len(missing_students)} students")
else:
    print("\n⚠️ No data to load")

# Verify
result = pd.read_sql("SELECT COUNT(DISTINCT uid) as students FROM dinning", engine)
print(f"✅ Total students now in dinning: {result['students'][0]}/49")

LOADING MISSING DINNING DATA
✓ Loaded u00: 7 entries
✓ Loaded u03: 2 entries
✓ Loaded u13: 0 entries
✓ Loaded u17: 3 entries
✓ Loaded u23: 1 entries
✓ Loaded u31: 0 entries
✓ Loaded u34: 0 entries
✓ Loaded u35: 2 entries
✓ Loaded u39: 0 entries
✓ Loaded u41: 0 entries
✓ Loaded u44: 3 entries
✓ Loaded u45: 1 entries
✓ Loaded u50: 0 entries
✓ Loaded u51: 4 entries
✓ Loaded u52: 2 entries
✓ Loaded u53: 3 entries
✓ Loaded u56: 2 entries
✓ Loaded u58: 15 entries


DatabaseError: Execution failed on sql 'INSERT INTO dinning (uid, datetime, location) VALUES (:uid, :datetime, :location)': (psycopg2.errors.UniqueViolation) duplicate key value violates unique constraint "dinning_pkey"
DETAIL:  Key (meal_id)=(1) already exists.

[SQL: INSERT INTO dinning (uid, datetime, location) VALUES (%(uid__0)s, %(datetime__0)s, %(location__0)s), (%(uid__1)s, %(datetime__1)s, %(location__1)s), (%(uid__2)s, %(datetime__2)s, %(location__2)s), (%(uid__3)s, %(datetime__3)s, %(location__3)s), (%(ui ... 1966 characters truncated ... (%(uid__43)s, %(datetime__43)s, %(location__43)s), (%(uid__44)s, %(datetime__44)s, %(location__44)s)]
[parameters: {'location__0': '43.70131169,-72.28938663', 'uid__0': 'u00', 'datetime__0': None, 'location__1': '43.71722994,-72.30901577', 'uid__1': 'u00', 'datetime__1': None, 'location__2': '43.70082504,-72.28929272', 'uid__2': 'u00', 'datetime__2': None, 'location__3': '43.70082924,-72.28928288', 'uid__3': 'u00', 'datetime__3': None, 'loca

In [4]:
# Check dinning table structure
result = pd.read_sql("SELECT * FROM dinning LIMIT 0", engine)
print("Dinning table columns:", result.columns.tolist())

Dinning table columns: ['meal_id', 'uid', 'datetime', 'location', 'meal_type']


In [5]:
# Check what the EMA Dining Halls data structure looks like
import json

dining_halls_dir = '/Users/test/Downloads/Data Management & Databases/studentlife_project/data/EMA/response/Dining Halls'
example_file = os.path.join(dining_halls_dir, 'Dining Halls_u00.json')

with open(example_file, 'r') as f:
    data = json.load(f)

print("Example Dining Halls entry:")
print(json.dumps(data[0], indent=2))

Example Dining Halls entry:
{
  "breakfast": "4",
  "dinner": "7",
  "location": "43.70131169,-72.28938663",
  "lunch": "4",
  "resp_time": 1367008780
}


In [9]:
# Fix the auto-increment sequence, then load data
from sqlalchemy import text

print("="*60)
print("FIXING DINNING AUTO-INCREMENT AND LOADING DATA")
print("="*60)

# Step 1: Find the current max meal_id
result = pd.read_sql("SELECT MAX(meal_id) as max_id FROM dinning", engine)
current_max = result['max_id'][0]
print(f"Current max meal_id: {current_max}")

# Step 2: Reset the sequence to start after current max
with engine.connect() as conn:
    conn.execute(text(f"ALTER SEQUENCE dinning_meal_id_seq RESTART WITH {current_max + 1}"))
    conn.commit()
print(f"✓ Reset sequence to start at {current_max + 1}")

# Step 3: Now load the data
import os
import json

dining_halls_dir = '/Users/test/Downloads/Data Management & Databases/studentlife_project/data/EMA/response/Dining Halls'
missing_students = ['u00', 'u03', 'u13', 'u17', 'u23', 'u31', 'u34', 'u35', 'u39', 'u41', 'u44', 'u45', 'u50', 'u51', 'u52', 'u53', 'u56', 'u58']

all_responses = []

for uid in missing_students:
    filename = f"Dining Halls_{uid}.json"
    filepath = os.path.join(dining_halls_dir, filename)
    
    if not os.path.exists(filepath):
        continue
    
    try:
        with open(filepath, 'r') as f:
            data = json.load(f)
        
        for entry in data:
            resp_time = entry.get('resp_time')
            location = entry.get('location')
            
            for meal_type in ['breakfast', 'lunch', 'dinner']:
                if meal_type in entry and entry[meal_type]:
                    all_responses.append((uid, resp_time, location, meal_type))
        
        print(f"✓ Loaded {uid}: {len(data)} surveys")
        
    except Exception as e:
        print(f"Error {uid}: {e}")

# Insert data
if all_responses:
    insert_query = text("""
        INSERT INTO dinning (uid, datetime, location, meal_type)
        VALUES (:uid, :datetime, :location, :meal_type)
    """)
    
    with engine.connect() as conn:
        for uid, datetime, location, meal_type in all_responses:
            conn.execute(insert_query, {
                'uid': uid,
                'datetime': datetime,
                'location': location,
                'meal_type': meal_type
            })
        conn.commit()
    
    print(f"\n✅ Inserted {len(all_responses):,} records")

# Verify
result = pd.read_sql("SELECT COUNT(DISTINCT uid) as students FROM dinning", engine)
print(f"✅ Total students: {result['students'][0]}/49")

FIXING DINNING AUTO-INCREMENT AND LOADING DATA
Current max meal_id: 7482
✓ Reset sequence to start at 7483
✓ Loaded u00: 7 surveys
✓ Loaded u03: 2 surveys
✓ Loaded u13: 0 surveys
✓ Loaded u17: 3 surveys
✓ Loaded u23: 1 surveys
✓ Loaded u31: 0 surveys
✓ Loaded u34: 0 surveys
✓ Loaded u35: 2 surveys
✓ Loaded u39: 0 surveys
✓ Loaded u41: 0 surveys
✓ Loaded u44: 3 surveys
✓ Loaded u45: 1 surveys
✓ Loaded u50: 0 surveys
✓ Loaded u51: 4 surveys
✓ Loaded u52: 2 surveys
✓ Loaded u53: 3 surveys
✓ Loaded u56: 2 surveys
✓ Loaded u58: 15 surveys

✅ Inserted 135 records
✅ Total students: 43/49


In [10]:
# Check which students are now in dinning table
result = pd.read_sql("SELECT DISTINCT uid FROM dinning ORDER BY uid", engine)
loaded_students = result['uid'].tolist()

print(f"Students in dinning table: {len(loaded_students)}")
print(f"UIDs: {loaded_students}")

# Compare to what files exist
dining_halls_dir = '/Users/test/Downloads/Data Management & Databases/studentlife_project/data/EMA/response/Dining Halls'
all_files = [f for f in os.listdir(dining_halls_dir) if f.endswith('.json')]
file_students = sorted([extract_uid(f) for f in all_files if extract_uid(f)])

print(f"\nStudents with files: {len(file_students)}")
missing_from_db = sorted(set(file_students) - set(loaded_students))

if missing_from_db:
    print(f"\nMissing from DB: {missing_from_db}")
    
    # Check if those files actually have data
    import json
    for uid in missing_from_db:
        filepath = os.path.join(dining_halls_dir, f"Dining Halls_{uid}.json")
        with open(filepath, 'r') as f:
            data = json.load(f)
        print(f"  {uid}: {len(data)} entries in file")
else:
    print(f"\n✅ ALL students with dining files are loaded!")

Students in dinning table: 43
UIDs: ['u00', 'u01', 'u02', 'u03', 'u04', 'u05', 'u07', 'u08', 'u09', 'u10', 'u12', 'u14', 'u15', 'u16', 'u17', 'u18', 'u19', 'u20', 'u22', 'u23', 'u24', 'u25', 'u27', 'u30', 'u32', 'u33', 'u35', 'u36', 'u42', 'u43', 'u44', 'u45', 'u46', 'u47', 'u49', 'u51', 'u52', 'u53', 'u54', 'u56', 'u57', 'u58', 'u59']

Students with files: 49

Missing from DB: ['u13', 'u31', 'u34', 'u39', 'u41', 'u50']
  u13: 0 entries in file
  u31: 0 entries in file
  u34: 0 entries in file
  u39: 0 entries in file
  u41: 0 entries in file
  u50: 0 entries in file
